### ЗАДАЧА: Доска задач поддержки

У команды поддержки есть входящий поток задач от клиентов.
Нужно собрать удобную модель, которая позволит быстро посмотреть:
- какие задачи ещё открыты,
- сколько часов планируется по каждому клиенту,
- у какого клиента сейчас самая большая нагрузка,
- как меняется состояние доски после закрытия одной из задач.

На входе у тебя есть несколько строк с данными о задачах.
На выходе должен получиться компактный отчёт по текущей доске.


In [ ]:
# rows: ticket_id|client|title|estimate_hours|status
rows = [
    'TK-100|Acme|Ошибка в отчёте|3.5|new',
    'TK-101|Beta|Починить интеграцию|5|in_progress',
    'TK-102|Acme|Обновить доступы|2|new',
    'TK-103|Delta|Проверить выгрузку|1.5|closed',
]

class Ticket:
    allowed_statuses = {'new', 'in_progress', 'closed'}

    def __init__(self, ticket_id, client, title, estimate_hours, status):
        self.ticket_id = ticket_id
        self.client = client
        self.title = title
        self.estimate_hours = estimate_hours
        if status not in self.allowed_statuses:
            raise ValueError(f"Статус {status} недопустим. Допустимые значения: {self.allowed_statuses}")
        self._status = status

    @property
    def estimate_hours(self):
        return self._estimate_hours

    @estimate_hours.setter
    def estimate_hours(self, value):
        value = float(value)
        if value <= 0:
            raise ValueError('Количество часов должно быть больше 0')
        self._estimate_hours = value

    def close(self):
        self._status = 'closed'

    def reopen(self):
        self._status = 'new'

    @classmethod
    def from_row(cls, row):
        parts = row.split('|')
        if len(parts) != 5:
            raise ValueError(f"В строке должно быть 5 элементов, фактически: {len(parts)}: {row}")
        ticket_id, client, title, estimate_hours, status = parts
        return cls(ticket_id, client, title, estimate_hours, status)

    @property
    def status(self):
        return self._status

    def __repr__(self):
        return f"Ticket(ticket_id='{self.ticket_id}', client='{self.client}', status='{self.status}')"



class TicketBoard:
    def __init__(self):
        self.tickets = []

    def add(self, ticket):
        self.tickets.append(ticket)

    def load(self, rows):
        for row in rows:
            ticket = Ticket.from_row(row)
            self.add(ticket)

    def open_tickets(self):
        return [ticket for ticket in self.tickets if ticket.status != 'closed']

    def by_client(self, client):
        return [ticket for ticket in self.tickets if ticket.client == client]

    def total_hours_by_client(self):
        hours_dict = {}
        for ticket in self.tickets:
            if ticket.client not in hours_dict:
                hours_dict[ticket.client] = 0
            hours_dict[ticket.client] += ticket.estimate_hours
        return hours_dict

    def busiest_client(self):
        total_hours = self.total_hours_by_client()
        if not total_hours:
            return None
        busiest = max(total_hours.items(), key=lambda x: x[1])
        return busiest

board = TicketBoard()
board.load(rows)

print("=== ОТЧЁТ ПО ДОСКЕ ЗАДАЧ ПОДДЕРЖКИ ===\n")

print("1. Все тикеты:")
for ticket in board.tickets:
    print(f"  {ticket}")
print()

print("2. Открытые тикеты (статус != 'closed'):")
open_tickets = board.open_tickets()
for ticket in open_tickets:
    print(f"  {ticket}")
print()

print("3. Тикеты клиента 'Acme':")
acme_tickets = board.by_client('Acme')
for ticket in acme_tickets:
    print(f"  {ticket}")
print()

print("4. Суммарные часы по клиентам:")
hours_by_client = board.total_hours_by_client()
for client, hours in hours_by_client.items():
    print(f"  {client}: {hours} ч.")
print()

busiest = board.busiest_client()
print(f"5. Самый загруженный клиент: {busiest[0]} ({busiest[1]} ч.)")
print()

if open_tickets:
    first_open = open_tickets[0]
    print(f"6. Закрываем первую открытую задачу: {first_open}")
    first_open.close()

    print("\n7. Открытые тикеты после закрытия первой задачи:")
    updated_open = board.open_tickets()
    for ticket in updated_open:
        print(f"  {ticket}")
else:
    print("6. Открытых задач нет.")

# TODO: загрузить строки в board
# TODO: вывести все тикеты
# TODO: вывести только открытые тикеты
# TODO: вывести задачи клиента 'Acme'
# TODO: вывести total_hours_by_client()
# TODO: вывести busiest_client()
# TODO: закрыть первую открытую задачу и снова вывести open_tickets()
